# 02 — Liquidity Calculation

Compute intraday liquidity metrics and visualise cumulative net positions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import load_raw_data, preprocess
from src.feature_engineering import build_features, get_feature_matrix

%matplotlib inline
sns.set_theme(style='whitegrid')


## 1. Preprocess data

In [ ]:
df_raw = load_raw_data('../data/raw/synthetic_intraday_payments_500k.csv')
df = preprocess(df_raw)
print(f'Processed shape: {df.shape}')
df.head()


## 2. Cumulative net position per bank

In [ ]:
daily_net = (
    df.groupby(['bank_id', 'date'])['signed_amount']
    .sum()
    .reset_index()
    .sort_values(['bank_id', 'date'])
)
daily_net['cumulative_net'] = daily_net.groupby('bank_id')['signed_amount'].cumsum()
print(daily_net.head(10))


## 3. Liquidity curve visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for bank in daily_net['bank_id'].unique()[:8]:
    bank_data = daily_net[daily_net['bank_id'] == bank]
    ax.plot(
        pd.to_datetime(bank_data['date']),
        bank_data['cumulative_net'],
        label=bank,
        linewidth=1.2,
    )

ax.set_title('Intraday Liquidity Curve – Cumulative Net Position', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Net Position (USD)')
ax.legend(title='Bank', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/liquidity_curve.png', dpi=150)
plt.show()


## 4. Peak vs off-peak liquidity

In [ ]:
# Peak hours: 09:00–12:00 (consistent with feature_engineering.py)
df['is_peak_hour'] = ((df['hour'] >= 9) & (df['hour'] < 12)).astype(int)

peak_vs_off = (
    df.groupby('is_peak_hour')['amount']
    .agg(['mean', 'sum', 'count'])
    .rename(index={0: 'Off-Peak', 1: 'Peak'})
)
print(peak_vs_off)


## 5. Daily net position heatmap

In [ ]:
pivot = daily_net.pivot_table(
    index='bank_id',
    columns='date',
    values='signed_amount',
    aggfunc='sum',
).iloc[:, :30]  # first 30 days

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(pivot, cmap='RdYlGn', center=0, ax=ax, linewidths=0)
ax.set_title('Daily Net Position per Bank (first 30 business days)')
plt.tight_layout()
plt.show()
